In [1]:
%load_ext autoreload
%autoreload 2

import sys, logging, importlib

# Ensure your scripts dir is importable
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/postprocessing")


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py
import pyarrow as pa
import pyarrow.parquet as pq
import logging
import awkward as ak
import uproot
import time

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch, utils

from pathlib import Path
import logging

# Reuse postprocessing helpers
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/postprocessing")
from convert_particles import build_particles_df_with_parents_and_vertex, write_particles_with_selection
from convert_digihits import process_event_for_digihits, write_digihits_with_selection
from convert_calorimeter import write_calohits_with_selection, process_calohits_batch
from utils.path_utils import get_run_paths, make_dir
from utils.track_utils import load_root_file, load_track_summary, create_particle_barcode_map

from convert_all import convert_all


## Roadmap

1. Load trackstates root file from v1
2. Load tracks_ambi.csv file from v1
3. Load trackstates root file from v2
4. Check that all info in v2 is present in v1

In [3]:
## Load v1

trackstates_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/trackstates_ambi.root"
tracksummary_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/tracksummary_ambi.root"
tracks_csv_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/event000000000-tracks_ambi.csv"
particles_v1 = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/particles.root"

In [8]:
track_summary_file = uproot.open(tracksummary_v1)
track_summary_file["tracksummary"].keys()


['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'nOutliers',
 'nHoles',
 'nSharedHits',
 'chi2Sum',
 'NDF',
 'measurementChi2',
 'outlierChi2',
 'measurementVolume',
 'measurementLayer',
 'outlierVolume',
 'outlierLayer',
 'nMajorityHits',
 'majorityParticleId',
 'trackClassification',
 't_charge',
 't_time',
 't_vx',
 't_vy',
 't_vz',
 't_px',
 't_py',
 't_pz',
 't_theta',
 't_phi',
 't_eta',
 't_p',
 't_pT',
 't_d0',
 't_z0',
 't_prodR',
 'hasFittedParams',
 'eLOC0_fit',
 'eLOC1_fit',
 'ePHI_fit',
 'eTHETA_fit',
 'eQOP_fit',
 'eT_fit',
 'err_eLOC0_fit',
 'err_eLOC1_fit',
 'err_ePHI_fit',
 'err_eTHETA_fit',
 'err_eQOP_fit',
 'err_eT_fit',
 'res_eLOC0_fit',
 'res_eLOC1_fit',
 'res_ePHI_fit',
 'res_eTHETA_fit',
 'res_eQOP_fit',
 'res_eT_fit',
 'pull_eLOC0_fit',
 'pull_eLOC1_fit',
 'pull_ePHI_fit',
 'pull_eTHETA_fit',
 'pull_eQOP_fit',
 'pull_eT_fit',
 'cov_eLOC0_eLOC0',
 'cov_eLOC0_eLOC1',
 'cov_eLOC0_ePHI',
 'cov_eLOC0_eTHETA',
 'cov_eLOC0_eQOP',
 'cov_eLOC0_eT',
 'cov_eLOC1_e

In [ ]:
track_fitting_file = uproot.open(trackstates_v1)
track_fitting_file["trackstates"].keys()

['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'volume_id',
 'layer_id',
 'module_id',
 'stateType',
 'chi2',
 'pathLength',
 't_x',
 't_y',
 't_z',
 't_r',
 't_dx',
 't_dy',
 't_dz',
 't_eLOC0',
 't_eLOC1',
 't_ePHI',
 't_eTHETA',
 't_eQOP',
 't_eT',
 'dim_hit',
 'l_x_hit',
 'l_y_hit',
 'g_x_hit',
 'g_y_hit',
 'g_z_hit',
 'res_x_hit',
 'res_y_hit',
 'err_x_hit',
 'err_y_hit',
 'pull_x_hit',
 'pull_y_hit',
 'nPredicted',
 'predicted',
 'eLOC0_prt',
 'eLOC1_prt',
 'ePHI_prt',
 'eTHETA_prt',
 'eQOP_prt',
 'eT_prt',
 'res_eLOC0_prt',
 'res_eLOC1_prt',
 'res_ePHI_prt',
 'res_eTHETA_prt',
 'res_eQOP_prt',
 'res_eT_prt',
 'err_eLOC0_prt',
 'err_eLOC1_prt',
 'err_ePHI_prt',
 'err_eTHETA_prt',
 'err_eQOP_prt',
 'err_eT_prt',
 'pull_eLOC0_prt',
 'pull_eLOC1_prt',
 'pull_ePHI_prt',
 'pull_eTHETA_prt',
 'pull_eQOP_prt',
 'pull_eT_prt',
 'g_x_prt',
 'g_y_prt',
 'g_z_prt',
 'px_prt',
 'py_prt',
 'pz_prt',
 'eta_prt',
 'pT_prt',
 'nFiltered',
 'filtered',
 'eLOC0_flt',
 'eLOC1_flt',
 'ePHI

In [9]:
particles_file = uproot.open(particles_v1)
particles_file["particles"].keys()


['event_id',
 'particle_hash',
 'particle_type',
 'process',
 'vx',
 'vy',
 'vz',
 'vt',
 'px',
 'py',
 'pz',
 'm',
 'q',
 'eta',
 'phi',
 'pt',
 'p',
 'q_over_p',
 'theta',
 'vertex_primary',
 'vertex_secondary',
 'particle',
 'generation',
 'sub_particle',
 'e_loss',
 'total_x0',
 'total_l0',
 'number_of_hits',
 'outcome']

In [ ]:
included_tracksummary_columns = ["event_nr", "track_nr", "eLOC0_fit", "eLOC1_fit", "ePHI_fit", "eTHETA_fit", "eQOP_fit", "eT_fit", "t_d0", "t_z0", "t_phi", "t_theta", "t_charge", "t_p", "t_pT", "t_time"]
track_summary_df = load_root_file(str(tracksummary_v1), included_columns=included_tracksummary_columns, events=(0,1))


In [5]:
included_trackstates_columns = ['event_nr',
 'track_nr',
 'nStates',
 'nMeasurements',
 'volume_id',
 'layer_id',
 'module_id',
 'stateType',
 'chi2',
 'pathLength',
 't_x',
 't_y',
 't_z',
 't_r',
 't_dx',
 't_dy',
 't_dz',
 't_eLOC0',
 't_eLOC1',
 't_ePHI',
 't_eTHETA',
 't_eQOP',
 't_eT',
 'dim_hit',
 'l_x_hit',
 'l_y_hit',
 'g_x_hit',
 'g_y_hit',
 'g_z_hit',
 'res_x_hit',
 'res_y_hit',
 'err_x_hit',
 'err_y_hit',
 'pull_x_hit',
 'pull_y_hit',
 'nPredicted',
 'predicted',
 'eLOC0_prt',
 'eLOC1_prt',
 'ePHI_prt',
 'eTHETA_prt',
 'eQOP_prt',
 'eT_prt',
 'res_eLOC0_prt',
 'res_eLOC1_prt',
 'res_ePHI_prt',
 'res_eTHETA_prt',
 'res_eQOP_prt',
 'res_eT_prt',
 'err_eLOC0_prt',
 'err_eLOC1_prt',
 'err_ePHI_prt',
 'err_eTHETA_prt',
 'err_eQOP_prt',
 'err_eT_prt',
 'pull_eLOC0_prt',
 'pull_eLOC1_prt',
 'pull_ePHI_prt',
 'pull_eTHETA_prt',
 'pull_eQOP_prt',
 'pull_eT_prt',
 'g_x_prt',
 'g_y_prt',
 'g_z_prt',
 'px_prt',
 'py_prt',
 'pz_prt',
 'eta_prt',
 'pT_prt',
 'nFiltered',
 'filtered',
 'eLOC0_flt',
 'eLOC1_flt',
 'ePHI_flt',
 'eTHETA_flt',
 'eQOP_flt',
 'eT_flt',
 'res_eLOC0_flt',
 'res_eLOC1_flt',
 'res_ePHI_flt',
 'res_eTHETA_flt',
 'res_eQOP_flt',
 'res_eT_flt',
 'err_eLOC0_flt',
 'err_eLOC1_flt',
 'err_ePHI_flt',
 'err_eTHETA_flt',
 'err_eQOP_flt',
 'err_eT_flt',
 'pull_eLOC0_flt',
 'pull_eLOC1_flt',
 'pull_ePHI_flt',
 'pull_eTHETA_flt',
 'pull_eQOP_flt',
 'pull_eT_flt',
 'g_x_flt',
 'g_y_flt',
 'g_z_flt',
 'px_flt',
 'py_flt',
 'pz_flt',
 'eta_flt',
 'pT_flt',
 'nSmoothed',
 'smoothed',
 'eLOC0_smt',
 'eLOC1_smt',
 'ePHI_smt',
 'eTHETA_smt',
 'eQOP_smt',
 'eT_smt',
 'res_eLOC0_smt',
 'res_eLOC1_smt',
 'res_ePHI_smt',
 'res_eTHETA_smt',
 'res_eQOP_smt',
 'res_eT_smt',
 'err_eLOC0_smt',
 'err_eLOC1_smt',
 'err_ePHI_smt',
 'err_eTHETA_smt',
 'err_eQOP_smt',
 'err_eT_smt',
 'pull_eLOC0_smt',
 'pull_eLOC1_smt',
 'pull_ePHI_smt',
 'pull_eTHETA_smt',
 'pull_eQOP_smt',
 'pull_eT_smt',
 'g_x_smt',
 'g_y_smt',
 'g_z_smt',
 'px_smt',
 'py_smt',
 'pz_smt',
 'eta_smt',
 'pT_smt',
 'nUnbiased',
 'unbiased',
 'eLOC0_ubs',
 'eLOC1_ubs',
 'ePHI_ubs',
 'eTHETA_ubs',
 'eQOP_ubs',
 'eT_ubs',
 'res_eLOC0_ubs',
 'res_eLOC1_ubs',
 'res_ePHI_ubs',
 'res_eTHETA_ubs',
 'res_eQOP_ubs',
 'res_eT_ubs',
 'err_eLOC0_ubs',
 'err_eLOC1_ubs',
 'err_ePHI_ubs',
 'err_eTHETA_ubs',
 'err_eQOP_ubs',
 'err_eT_ubs',
 'pull_eLOC0_ubs',
 'pull_eLOC1_ubs',
 'pull_ePHI_ubs',
 'pull_eTHETA_ubs',
 'pull_eQOP_ubs',
 'pull_eT_ubs',
 'g_x_ubs',
 'g_y_ubs',
 'g_z_ubs',
 'px_ubs',
 'py_ubs',
 'pz_ubs',
 'eta_ubs',
 'pT_ubs']
track_states_df = load_root_file(str(trackstates_v1), included_columns=included_trackstates_columns, events=(0,1))

In [6]:
track_summary_df

event_id  track_nr    t_charge      t_time   t_theta  \
entry subentry                                                         
0     0                0         0  2147483647         NaN       NaN   
      1                0         1           1  132.644058  0.443062   
      2                0         2           1  132.644058  0.120443   
      3                0         3          -1  132.644058  0.105681   
      4                0         4          -1  132.644058  0.386801   
      5                0         5           1  132.644058  1.924163   
      6                0         6           1  136.470428  2.073422   
      7                0         7           1  139.962494  2.031866   
      8                0         8          -1  139.962494  1.963666   
      9                0         9          -1  133.848083  1.890744   
      10               0        10          -1  136.470428  2.047745   
      11               0        11          -1  139.962494  1.940108   
      12               0        12           1  133.848083  1.891127   
      13               0        13           1  132.644058  0.602277   
      14               0        14  2147483647         NaN       NaN   
      15               0        15           1  132.644058  1.767142   
      16               0        16           1  132.644058  1.978410   
      17               0        17           1  132.644058  2.257941   

                   t_phi         t_p       t_pT      t_d0        t_z0  \
entry subentry                                                          
0     0              NaN         NaN        NaN       NaN         NaN   
      1        -2.707376    3.010472   1.290613  0.007206  196.084473   
      2        -2.062113   37.637253   4.522196  0.009821  196.079025   
      3        -1.999850   11.193091   1.180694  0.009868  196.074493   
      4        -1.520458  113.981796  42.997116  0.008964  196.060043   
      5         0.656169    1.556274   1.460116 -0.008517  196.072083   
      6         0.779880    5.893669   5.164744  0.049872  196.401764   
      7         0.871462    4.365304   3.909466 -0.387924  196.461197   
      8         0.819515    3.000289   2.771710 -0.040825  195.916016   
      9         0.795814    7.218099   6.851793  0.008973  195.967056   
      10        0.785010    5.127962   4.555681  0.032098  196.287598   
      11        0.766632    5.523000   5.150618  0.312640  195.730423   
      12        0.933310    3.307349   3.139109 -0.142638  195.964478   
      13        1.323771   87.427788  49.529598 -0.009787  196.072189   
      14             NaN         NaN        NaN       NaN         NaN   
      15        1.584328    2.061903   2.022285 -0.009111  196.069473   
      16        1.702805    1.703632   1.564052 -0.008596  196.068130   
      17        3.061142    1.317205   1.018279  0.002946  196.062500   

                eLOC0_fit   eLOC1_fit  ePHI_fit  eTHETA_fit  eQOP_fit  \
entry subentry                                                          
0     0          0.032012  196.103989 -2.826225    2.153924 -0.436399   
      1         -0.025235  196.151077 -2.705781    0.443445  0.333322   
      2          0.026036  195.821991 -2.062616    0.120338  0.025388   
      3         -0.179633  195.763412 -1.994134    0.105620 -0.085778   
      4          0.042841  196.084808 -1.521513    0.386896 -0.020768   
      5         -0.013372  196.057205  0.656637    1.923691  0.651854   
      6          0.088478  196.403442  0.779719    2.073534  0.168209   
      7         -0.383887  196.466614  0.872830    2.031986  0.228828   
      8         -0.041257  195.916397  0.817460    1.963739 -0.332599   
      9          0.024662  195.979019  0.795247    1.890983 -0.139019   
      10         0.024997  196.292984  0.784518    2.047968 -0.195168   
      11         0.323651  195.709442  0.765203    1.939661 -0.182497   
      12        -0.072106  195.929321  0.931239    1.890278  0.309090   
      13        -0.

In [7]:
print(track_states_df.columns)

Index(['event_id', 'track_nr', 'nStates', 'nMeasurements', 'volume_id',
       'layer_id', 'module_id', 'stateType', 'chi2', 'pathLength',
       ...
       'pull_eQOP_ubs', 'pull_eT_ubs', 'g_x_ubs', 'g_y_ubs', 'g_z_ubs',
       'px_ubs', 'py_ubs', 'pz_ubs', 'eta_ubs', 'pT_ubs'],
      dtype='object', length=171)


### CSV check

In [4]:
tracks_csv = pd.read_csv(tracks_csv_v1)

In [6]:
tracks_csv

,track_id,seed_id,particleId,nStates,nMajorityHits,nMeasurements,nOutliers,nHoles,nSharedHits,chi2,ndf,chi2/ndf,pT,eta,phi,truthMatchProbability,good/duplicate/fake,Measurements_ID
0,16,29,vp=1|vs=0|p=0|g=1|sp=150,27,15,15,0,0,0,15.5426,26,0.597792,1.55393,-0.418693,1.701410,1.00000,good,"[124,251,255,279,401,406,732,826,938,1027,1026..."
1,15,28,vp=1|vs=0|p=0|g=1|sp=154,25,13,13,0,0,0,30.9455,22,1.406610,2.02290,-0.196890,1.584390,1.00000,good,"[123,250,270,397,731,827,821,937,1021,1320,132..."
2,1,1,vp=1|vs=0|p=0|g=0|sp=30,24,11,11,0,0,0,10.6233,18,0.590183,1.28720,1.489750,-2.705780,1.00000,good,"[145,175,306,759,867,1079,1109,1552,1553,1574,..."
3,2,2,vp=1|vs=0|p=0|g=0|sp=36,16,8,8,0,0,0,14.2295,16,0.889342,4.72857,2.809390,-2.062620,1.00000,good,"[79,460,477,496,518,565,600,641,]"
4,3,4,vp=1|vs=0|p=0|g=1|sp=110,25,15,15,0,0,0,27.0792,30,0.902639,1.22903,2.940130,-1.994130,1.00000,good,"[80,461,459,478,476,498,495,525,517,554,548,60..."
5,4,5,vp=1|vs=0|p=0|g=0|sp=13,28,13,13,0,0,0,20.9450,22,0.952046,18.16820,1.630160,-1.521510,1.00000,good,"[84,322,323,774,886,885,1081,1110,1136,1568,15..."
6,5,7,vp=1|vs=0|p=0|g=1|sp=10,26,13,11,3,0,0,26.5955,20,1.329770,1.43955,-0.360456,0.656637,1.18182,good,"[71,102,216,332,452,453,791,895,979,1058,1363,..."
7,6,8,vp=1|vs=18|p=0|g=2|sp=11,25,13,13,0,0,0,17.6611,22,0.802778,5.20938,-0.525360,0.779719,1.00000,good,"[105,230,341,455,387,793,902,1008,1009,1390,13..."
8,7,9,vp=1|vs=0|p=0|g=3|sp=6,26,14,14,0,0,0,23.4133,24,0.975554,3.91352,-0.478465,0.872830,1.00000,good,"[103,228,233,351,385,390,798,901,920,1012,1389..."
9,8,10,vp=1|vs=2|p=0|g=2|sp=9,28,16,16,0,0,0,14.8858,26,0.572530,2.77748,-0.403463,0.817460,1.00000,good,"[101,226,357,384,389,801,797,913,925,1017,1403..."
